# 08 · Capstone: build DETR from scratch in 50 lines

> **This repo's README:** *"Inference in 50 lines of PyTorch."*

Time to collect. You've taken DETR apart across seven notebooks — now **write it yourself, from a blank cell**, load the real pretrained weights into your own class, and detect real objects.

If your hand-built model produces the same cats and remotes as the official one, you understand DETR.

**The rule for this notebook:** we import `torch`, `nn`, and a ResNet. Nothing from `models/`.

Every operation you need is in [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) — §14 is this model's whole forward pass written out as shapes. Keep it open in a second tab.

## 0. Setup — images, boxes and plotting

Only third-party packages plus a few helpers written out here. **No model code** in this section: building the model is the whole point of this notebook.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names and plot colors

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

DETR predicts boxes as **`cxcywh`** — centre + size, normalized to `[0, 1]`. IoU and plotting want **`xyxy`** corners. Mixing the two up is the single most common bug in detection code, so both conversions live here.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

### Images in, tensors out

DETR's eval transform resizes the shortest side to 800px and ImageNet-normalizes. There is no fixed crop — the model accepts any input size.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

### Drawing detections

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

In [ ]:
print("ready — now go build a detector")

## 1. What we need to build

From notebook `02` §6, the full pipeline:

```
image ─► ResNet-50 ─► 1x1 conv ─► flatten +pos ─► Transformer ─► 2 heads ─► (class, box)
        (2048ch)      (256ch)     (HW,B,256)      enc+dec        per query
```

Every piece is either `torchvision` or `torch.nn`. Let's enumerate what we must define:

| Piece | Module | Explained in |
|---|---|---|
| CNN backbone | `resnet50()`, drop the `fc` head | `03` §2 |
| channel squeeze | `nn.Conv2d(2048, 256, 1)` | `03` §3 |
| positional encoding | learned row/col embeddings | `03` §5 |
| encoder + decoder | `nn.Transformer` | `01`, `04` |
| object queries | `nn.Parameter(100, 256)` | `04` §2 |
| class head | `nn.Linear(256, 91+1)` | `04` §4 |
| box head | `nn.Linear(256, 4)` + `sigmoid` | `04` §4 |

**Simplification:** we'll use `nn.Transformer` (PyTorch's stock implementation) instead of the repo's custom one, and *learned* position embeddings instead of sine. Both are choices the paper explicitly allows — Table 3 shows learned encodings score about the same.

## 2. Write it

Read the comments — each maps a line back to the notebook that explained it.

In [ ]:
class DETRdemo(nn.Module):
    """Minimal DETR. Inference only -- no masks, no aux losses.

    Deliberately simplified vs models/detr.py; see §5 for the exact diff.
    """

    def __init__(self, num_classes, hidden_dim=256, nheads=8,
                 num_encoder_layers=6, num_decoder_layers=6):
        super().__init__()

        # --- backbone (nb 03 §2): ResNet-50 minus its classifier ---
        self.backbone = torchvision.models.resnet50()
        del self.backbone.fc                       # we only want the feature map

        # --- channel squeeze (nb 03 §3): 2048 -> 256, per-pixel ---
        self.conv = nn.Conv2d(2048, hidden_dim, 1)

        # --- the transformer (nb 01, nb 04): 6 encoder + 6 decoder layers ---
        self.transformer = nn.Transformer(
            hidden_dim, nheads, num_encoder_layers, num_decoder_layers)

        # --- prediction heads (nb 04 §4) ---
        self.linear_class = nn.Linear(hidden_dim, num_classes + 1)   # +1 for "no object"
        self.linear_bbox = nn.Linear(hidden_dim, 4)                  # cx, cy, w, h

        # --- object queries (nb 04 §2): 100 learned vectors, image-independent ---
        self.query_pos = nn.Parameter(torch.rand(100, hidden_dim))

        # --- learned positional encoding (nb 03 §5): half the dims for rows,
        #     half for columns; 50 is just a max supported grid size ---
        self.row_embed = nn.Parameter(torch.rand(50, hidden_dim // 2))
        self.col_embed = nn.Parameter(torch.rand(50, hidden_dim // 2))

    def forward(self, inputs):
        # inputs shape: (B, 3, H0, W0)
        b = self.backbone
        x = b.maxpool(b.relu(b.bn1(b.conv1(inputs))))
        x = b.layer4(b.layer3(b.layer2(b.layer1(x))))     # (B, 2048, H0/32, W0/32)

        h = self.conv(x)                                   # (B, 256, H, W)
        H, W = h.shape[-2:]

        # build a (H*W, 1, 256) positional encoding by pairing every row with every col
        pos = torch.cat([
            self.col_embed[:W].unsqueeze(0).repeat(H, 1, 1),   # (H, W, 128)
            self.row_embed[:H].unsqueeze(1).repeat(1, W, 1),   # (H, W, 128)
        ], dim=-1).flatten(0, 1).unsqueeze(1)                  # (H*W, 1, 256)

        # flatten the grid into a sequence (nb 03 §4) and add position
        src = pos + 0.1 * h.flatten(2).permute(2, 0, 1)        # (H*W, B, 256)
        tgt = self.query_pos.unsqueeze(1)                      # (num_queries, B, d_model)

        h = self.transformer(src, tgt).transpose(0, 1)         # (B, 100, 256)

        return {"pred_logits": self.linear_class(h),           # (B, 100, 92)
                "pred_boxes": self.linear_bbox(h).sigmoid()}   # (B, 100, 4)


# NOTE: inspect.getsource() cannot read a class defined in a notebook cell
# ("source code not available"), so we just count the sub-modules instead.
print("DETRdemo is defined. Ignoring comments and blank lines, the class above is")
print("under 30 logical lines -- the README's '50 lines' was not an exaggeration.")

In [ ]:
torch.tensor([[1,2,3], [1,2,3]]).unsqueeze(2).shape

## 3. Load the real weights

Facebook published a checkpoint for exactly this simplified architecture. If our class is right, it loads with **zero missing and zero unexpected keys** — a strict structural test of our implementation.

In [ ]:
detr = DETRdemo(num_classes=91)
state = torch.hub.load_state_dict_from_url(
    "https://dl.fbaipublicfiles.com/detr/detr_demo-da2a99e9.pth",
    map_location="cpu", check_hash=True)

missing, unexpected = detr.load_state_dict(state, strict=False)
print("missing keys   :", len(missing))
print("unexpected keys:", len(unexpected))
assert not missing and not unexpected, "architecture mismatch!"
print("\nPERFECT MATCH -- your class has exactly the right structure.")
detr.eval()
print(f"parameters: {sum(p.numel() for p in detr.parameters())/1e6:.1f}M")

That assertion is the real test. Every parameter name, shape, and nesting level in your class had to match the checkpoint. Get `hidden_dim` wrong, forget the `+1` on the class head, or nest a module differently, and it fails.

## 4. Detect

In [ ]:
# our own little inference helper for DETRdemo
@torch.no_grad()
def detect_demo(model, pil, threshold=0.7):
    x = default_transform(pil).unsqueeze(0)              # (B, C, H, W) = (1, 3, H, W)
    out = model(x)
    probs = out["pred_logits"].softmax(-1)[0, :, :-1]    # drop the no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(out["pred_boxes"][0, keep], pil.size)
    return probs[keep], boxes


im = load_image("cats")
probs, boxes = detect_demo(detr, im)

print(f"{len(probs)} detections from YOUR model:\n")
for p, b in zip(probs, boxes):
    print(f"   {COCO_CLASSES[p.argmax()]:<8} {p.max():.3f}   {[round(v) for v in b.tolist()]}")

plot_results(im, probs, boxes, title="Detections from a model you wrote yourself")
plt.show()

Two cats, two remotes, a couch — from **your** class, written from a blank cell.

### Does it agree with the official model?

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048, aux_loss=False):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(), PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)   # +1 = "no object"
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries
        self.aux_loss = aux_loss                # keep every decoder layer's prediction

    def forward(self, images, mask=None):
        """images: (B, 3, H, W). mask: (B, H, W) with True on padded pixels."""
        if mask is None:                        # a single image needs no padding
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        # hs: (num_decoder_layers, B, num_queries, hidden_dim)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)

        outputs_class = self.class_embed(hs)             # (layers, B, queries, classes+1)
        outputs_coord = self.bbox_embed(hs).sigmoid()    # (layers, B, queries, 4)
        out = {"pred_logits": outputs_class[-1], "pred_boxes": outputs_coord[-1]}
        if self.aux_loss:
            out["aux_outputs"] = [{"pred_logits": a, "pred_boxes": b}
                                  for a, b in zip(outputs_class[:-1], outputs_coord[:-1])]
        return out


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None, aux_loss=False):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR(aux_loss=aux_loss)
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep


official = load_pretrained_detr(device="cpu")
op, ob, _, _ = detect(official, im, threshold=0.7)   # detect() from the block above

mine = sorted(COCO_CLASSES[p.argmax()] for p in probs)
theirs = sorted(COCO_CLASSES[p.argmax()] for p in op)
print("my 50-line model :", mine)
print("official model   :", theirs)
print("same objects     :", mine == theirs)
print()
print("(Boxes differ by a few pixels: different position encodings — learned vs sine —")
print(" and different checkpoints. Same architecture, separately trained.)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_results(im, probs, boxes, ax=axes[0], title="yours (50 lines, nn.Transformer)")
plot_results(im, op, ob, ax=axes[1], title="official (models/detr.py)")
plt.tight_layout(); plt.show()

## 5. What the 50-line version leaves out

Being explicit about the gap is the point — each omission is a feature you now know the purpose of.

| Dropped | Why the full version needs it | Notebook |
|---|---|---|
| `NestedTensor` + padding masks | batching images of different sizes without attending to padding | `03` §1 |
| Sine positional encoding | generalizes past the hard-coded 50×50 grid; +1.4 AP over input-only | `03` §5 |
| Position re-added at **every** attention layer | `nn.Transformer` only adds it once at the input; Table 3 ablation | `03` §5 |
| `bbox_embed` as a 3-layer MLP | here it's one `Linear` — the MLP regresses geometry better | `04` §4 |
| `return_intermediate_dec` | all 6 decoder layers, needed for auxiliary losses | `04` §3 |
| `SetCriterion` + `HungarianMatcher` | **all of training.** Inference needs none of it | `05` |
| `FrozenBatchNorm2d` | stable training with 2 images per GPU | `03` §2 |

Notice the shape of that list: nearly everything dropped is about **training** or **batching**. The core *idea* — CNN → transformer → N queries → N predictions — really does fit in 50 lines. That is the paper's claim in §3.2:

> *"DETR can be implemented in less than 50 lines in PyTorch."*

## 6. Exercises

**Exercise 1.** Change `hidden_dim` to 128 and reload the checkpoint. What error, and why?

<details><summary>Solution</summary>

A shape-mismatch `RuntimeError` on nearly every parameter: `copying a param with shape torch.Size([256, 2048, 1, 1]) from checkpoint, the shape in current model is torch.Size([128, 2048, 1, 1])`.

`hidden_dim` is the transformer's `d_model` — it fixes the width of *every* weight downstream. It isn't a free knob for a pretrained checkpoint; changing it means retraining.
</details>

---

**Exercise 2.** Remove `.sigmoid()` from `linear_bbox`. Predictions still appear — why are the boxes wrong?

<details><summary>Solution</summary>

Box coordinates are **normalized `(cx, cy, w, h)` in `[0,1]`** (nb `02` §5). Without sigmoid the head emits unbounded reals, so `rescale_bboxes` maps them far outside the image — and negative widths make `Rectangle` degenerate.

Sigmoid isn't cosmetic: it encodes the constraint that a box lies inside the image.
</details>

---

**Exercise 3.** What happens with `torch.rand(100, hidden_dim)` replaced by `torch.zeros(100, hidden_dim)` for `query_pos` (before loading weights)?

<details><summary>Solution</summary>

At *init* it doesn't matter — `load_state_dict` overwrites it with trained values.

But if you zero it out **after** loading, all 100 queries become identical, and by permutation-invariance (nb `01` §5) they produce 100 identical predictions. Try it:

```python
detr.query_pos.data.zero_()
p, b = detect(detr, im, threshold=0.0)
print("unique boxes:", len(torch.unique(b, dim=0)))   # -> 1
```
</details>

---

**Exercise 4 (harder).** The model hard-codes a 50×50 max grid via `row_embed`/`col_embed`. At what input resolution does it break, and what error do you get?

<details><summary>Solution</summary>

The grid is `ceil(size/32)` (nb `03`), so it breaks past `50 × 32 = 1600px` on a side. `self.col_embed[:W]` silently returns fewer than `W` rows, and the `torch.cat` then fails on mismatched dims.

```python
big = torch.randn(1, 3, 800, 1700)     # 1700/32 = 54 > 50
try:
    detr(big)
except RuntimeError as e:
    print("RuntimeError:", str(e)[:120])
```

This is exactly why the full DETR uses **sine** encodings — they're computed from coordinates, so they work at any resolution with no lookup table.
</details>

## You've finished the tutorial

You built DETR. Everything in [`models/detr.py`](../models/detr.py) is now either something you wrote yourself or something you know the reason for.

**Go read the real source now** — [`models/detr.py`](../models/detr.py), [`models/transformer.py`](../models/transformer.py), [`models/matcher.py`](../models/matcher.py). It should read like a slightly more careful version of what you just wrote.

**Then re-read the paper.** It'll feel like reviewing notes rather than learning something new.

### Where to go next

| Direction | Why |
|---|---|
| **Deformable DETR** | fixes the slow convergence you hit in `07` and the weak small-object AP from `03` |
| **Panoptic segmentation** | [`models/segmentation.py`](../models/segmentation.py) in this repo; paper §4.4 |
| **Train on real data** | notebook `07` §8 has the two things you need |
| **DINO / DETA / RT-DETR** | the modern descendants; all assume you know this paper |